# AskSight — mixed-data model experiment (Kaggle / Colab GPU)

`photo → pretrained CLIP → trained connector → frozen Qwen2.5 → answer`

This notebook trains the AskSight model with VQA questions plus COCO image descriptions. It is currently set for **Run B**: sentence-style answers, connector only, one caption per row.

## Kaggle quick start

1. Push the current AskSight changes to the `dev` branch; the install cell clones that branch.
2. Upload this notebook, enable **GPU** and **Internet**, then run the cells in order.
3. For Run B, run sections 0–5a. **Do not use “Run all”**: skip 5c (Qwen baseline) and 5d (robustness).
4. Download the checkpoint after evaluation; the trainer saves only at the end.

For **Run C**, change `VERSION` to `v0.4-c-lora16` and `LORA_RANK` to `16` in section 0, then rerun sections 4, 5, and 5a. The prepared data is reused. After both runs, section **5b** compares them against v0.3.


## 0. Configuration

Current setting: **Run B** — sentence targets, connector only, one caption per row.

- `TRAIN / VAL / TEST` — fixed to 20,000/1,000/1,000 VQA examples.
- `CAPTION_IMAGES` — 2,000 COCO train images, usually about 10,000 captions.
- `CAPTION_VAL_IMAGES` — 200 separate COCO train images for description validation.
- `CAPTION_SENTENCES` — combine up to N distinct captions per image (0 keeps one row per caption).
- `ANSWER_STYLE` — `sentence` restates common VQA answers as complete sentences; `short` reproduces v0.3.
- `LORA_RANK` — `None` trains only the connector (A/B); `16` trains connector + LoRA adapters (C).
- `DTYPE` — `float16` works on common Kaggle and Colab GPUs.
- `SEED` — fixes the data samples and split.

Keep the same split, prompt, and evaluation set when comparing experiments. The trainer saves only after both epochs complete.


In [ ]:
VERSION = "v0.4-b-sentence"    # Run B: sentence targets, connector only
TRAIN, VAL, TEST = 20_000, 1_000, 1_000
CAPTION_IMAGES, CAPTION_VAL_IMAGES = 2000, 200
CAPTION_SENTENCES = 0          # 0 = one caption per row, same as v0.3
ANSWER_STYLE = "sentence"      # full sentences; "short" is the v0.3 baseline
LORA_RANK = None               # None = connector only; 16 = Run C
DTYPE = "float16"
SEED = 0


## 1. Install (~5 min)

Clones the `dev` branch and installs the `asksight_model` package with LoRA support. Re-run safe; cached downloads are reused.


In [ ]:
import os
import subprocess
REPO = "https://github.com/Pranjal6955/AskSight.git"
BRANCH = "dev"
WORKSPACE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_DIR = os.path.join(WORKSPACE, "AskSight-training-dev")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR, check=True)
%cd {REPO_DIR}/model
for required in (
    "asksight_model/robustness.py",
    "asksight_model/coco_captions.py",
    "asksight_model/evaluate_vlm.py",
    "evaluations/asksight_v1.jsonl",
    "asksight_model/targets.py",
):
    assert os.path.isfile(required), (
        f"Missing {required} on {BRANCH}. Push the latest AskSight changes to GitHub, "
        "restart this runtime, and rerun the notebook."
    )
!pip install -q -e ".[dev,vlm_eval,lora]"


## 2. Verify GPU

Asserts CUDA is available; prints the assigned GPU (expect `Tesla T4`). If it fails: **Runtime → Change runtime type → T4 GPU**, then re-run.

In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU accelerator in the notebook Settings first"
import asksight_model
print("asksight_model", asksight_model.__name__, "ok")


## 3. Download VQA data

Downloads official VQA v2 questions/annotations plus the COCO images needed for this split. The split is deterministic and image-disjoint: images in train are kept separate from validation and test. The fingerprints are saved in `data/vqa/meta.json`.

For 50k, change the counts in cell 2 before running this cell. Actual counts can be slightly higher because the splitter keeps all questions for selected images together.


In [ ]:
%%time
!python -m asksight_model.data --out data/vqa --train {TRAIN} --val {VAL} --test {TEST} --seed {SEED} --answer-style {ANSWER_STYLE}
!cat data/vqa/meta.json


## 3b. Add COCO descriptions to training

The next cell samples 2,000 COCO train2014 images for training and 200 different images for caption validation. It combines up to `CAPTION_SENTENCES` distinct captions per image into `Describe this photo.` targets and downloads only selected images. The training and validation image IDs are disjoint from each other and the VQA validation/test image IDs. This is a first mixed-data candidate, not evidence that descriptions are grounded on AskSight's real user photos.

Set `EXTRA_EXAMPLES_FILE` to an optional reviewed JSONL (OCR/signs, unanswerable photos, real AskSight questions) with `question_id`, `image_id`, `image`, `question`, and `answer`. Images must be uploaded under `data/vqa/`. Do not include validation/test images.


In [ ]:
# Mix VQA questions with COCO descriptions (and optional reviewed examples). Safe to rerun.
import hashlib
import json
import subprocess
from pathlib import Path

CUSTOM_EXAMPLES_FILE = "data/asksight_coco_captions.jsonl"
CAPTION_VAL_FILE = "data/asksight_coco_captions_val.jsonl"
EXTRA_EXAMPLES_FILE = ""  # optional reviewed OCR/abstain/real-photo JSONL
prep_result = subprocess.run([
    "python", "-m", "asksight_model.coco_captions",
    "--data-dir", "data/vqa",
    "--out", CUSTOM_EXAMPLES_FILE,
    "--num-images", str(CAPTION_IMAGES),
    "--val-images", str(CAPTION_VAL_IMAGES),
    "--val-out", CAPTION_VAL_FILE,
    "--sentences-per-image", str(CAPTION_SENTENCES),
    "--seed", str(SEED),
], text=True, capture_output=True)
if prep_result.stdout:
    print(prep_result.stdout)
if prep_result.returncode:
    if prep_result.stderr:
        print(prep_result.stderr)
    raise subprocess.CalledProcessError(
        prep_result.returncode, prep_result.args, output=prep_result.stdout, stderr=prep_result.stderr
    )

examples_path = Path(CUSTOM_EXAMPLES_FILE)
caption_val_path = Path(CAPTION_VAL_FILE)
if not examples_path.is_file() or not caption_val_path.is_file():
    raise FileNotFoundError("COCO caption preparation did not produce both JSONL files")

data_dir = Path("data/vqa")
train_path = data_dir / "train.jsonl"
val_path = data_dir / "val.jsonl"
records = [json.loads(line) for line in train_path.read_text().splitlines() if line.strip()]
records = [
    row for row in records
    if row.get("source") not in {"asksight_coco_caption_train", "asksight_custom_train"}
]
vqa_train_images = {str(record["image_id"]) for record in records}
val_records = [json.loads(line) for line in val_path.read_text().splitlines() if line.strip()]
val_records = [row for row in val_records if row.get("source") != "asksight_coco_caption_val"]
held_out_images = {str(row["image_id"]) for row in val_records}
train_caption_rows = [
    json.loads(line) for line in examples_path.read_text().splitlines() if line.strip()
]
val_caption_rows = [
    json.loads(line) for line in caption_val_path.read_text().splitlines() if line.strip()
]
extra_rows = []
if EXTRA_EXAMPLES_FILE:
    extra_path = Path(EXTRA_EXAMPLES_FILE)
    if not extra_path.is_file():
        raise FileNotFoundError(extra_path)
    extra_rows = [json.loads(line) for line in extra_path.read_text().splitlines() if line.strip()]
    for row in extra_rows:
        row["source"] = "asksight_custom_train"
train_caption_rows.extend(extra_rows)
train_caption_images = {str(row["image_id"]) for row in train_caption_rows}
val_caption_images = set()
existing_ids = {str(row["question_id"]) for row in records}

for row_number, row in enumerate(train_caption_rows, start=1):
    for field in ("question_id", "image_id", "image", "question", "answer"):
        if field not in row or not str(row[field]).strip():
            raise ValueError(f"train example {row_number} needs a non-empty {field!r}")
    row["question_id"] = str(row["question_id"])
    row["image_id"] = str(row["image_id"])
    if row["image_id"] in held_out_images or row["question_id"] in existing_ids:
        raise ValueError(f"train example {row_number} overlaps held-out data or question IDs")
    if not (data_dir / row["image"]).is_file():
        raise FileNotFoundError(f"train image not found: {data_dir / row['image']}")
    if row.get("source") != "asksight_custom_train":
        row["source"] = "asksight_coco_caption_train"
    existing_ids.add(row["question_id"])
    records.append(row)

for row_number, row in enumerate(val_caption_rows, start=1):
    row["question_id"] = str(row["question_id"])
    row["image_id"] = str(row["image_id"])
    if (
        row["image_id"] in held_out_images
        or row["image_id"] in vqa_train_images
        or row["image_id"] in train_caption_images
    ):
        raise ValueError(f"caption validation row {row_number} has image leakage")
    if row["question_id"] in existing_ids:
        raise ValueError(f"caption validation row {row_number} duplicates a question ID")
    if not (data_dir / row["image"]).is_file():
        raise FileNotFoundError(f"caption image not found: {data_dir / row['image']}")
    row["source"] = "asksight_coco_caption_val"
    existing_ids.add(row["question_id"])
    val_caption_images.add(row["image_id"])
    val_records.append(row)

train_path.write_text("".join(json.dumps(row) + "\n" for row in records))
val_path.write_text("".join(json.dumps(row) + "\n" for row in val_records))
meta_path = data_dir / "meta.json"
meta = json.loads(meta_path.read_text())
extra_path = Path(EXTRA_EXAMPLES_FILE) if EXTRA_EXAMPLES_FILE else None
meta["custom_examples"] = {
    "train_file": examples_path.name,
    "train_sha256": hashlib.sha256(examples_path.read_bytes()).hexdigest(),
    "added_to_train": len(train_caption_rows) - len(extra_rows),
    "extra_file": extra_path.name if extra_path else None,
    "extra_sha256": hashlib.sha256(extra_path.read_bytes()).hexdigest() if extra_path else None,
    "added_extra": len(extra_rows),
    "validation_file": caption_val_path.name,
    "validation_sha256": hashlib.sha256(caption_val_path.read_bytes()).hexdigest(),
    "added_to_val": len(val_caption_rows),
    "train_images": len(train_caption_images),
    "validation_images": len(val_caption_images),
}
meta["actual"]["train"] = len(records)
meta["actual"]["val"] = len(val_records)
for split_name, split_rows in (("train", records), ("val", val_records)):
    meta["split_fingerprints"][split_name] = hashlib.sha256(
        ",".join(sorted(str(row["question_id"]) for row in split_rows)).encode()
    ).hexdigest()
meta_path.write_text(json.dumps(meta, indent=2) + "\n")
print(
    f"Mixed data: {len(records)} train rows ({ANSWER_STYLE} targets, {CAPTION_SENTENCES} captions/image); "
    f"{len(val_records)} val rows"
)


## 4. Train the candidate

This run keeps the VQA task and adds image descriptions. The checkpoint stores the exact prompt, answer targets, and LoRA setting used here; serving reads them back. `LORA_RANK = None` is experiment A/B (connector only); `LORA_RANK = 16` is experiment C (connector + LoRA). Use a distinct `VERSION` for each candidate.


In [ ]:
%%time
lora_flag = f"--lora-rank {LORA_RANK}" if LORA_RANK else ""
!python -m asksight_model.train --config configs/default.yaml --dtype {DTYPE} --version {VERSION} {lora_flag}


## 5. Evaluate the candidate

Run the standard VQA test plus the fixed AskSight set. `vqa_accuracy` is exact-match and drops for sentence answers, so compare B and C on `vqa_accuracy_contains` and `sentence_rate`. Description rows carry factual rubrics and are scored with a heuristic `fact_recall`; manual review is still required. The question-set hash, prompt, and predictions are saved with the checkpoint.


In [ ]:
%%time
!python -m asksight_model.evaluate --checkpoint checkpoints/{VERSION} --split test --data-dir data/vqa
import json
ev = json.load(open(f"checkpoints/{VERSION}/metrics.json"))["eval_test"]
print(
    f"\nVQA exact {ev['vqa_accuracy']:.1%} | contains {ev['vqa_accuracy_contains']:.1%} | "
    f"sentence rate {ev['sentence_rate']:.1%} | latency median {ev['latency_median_s']}s, p95 {ev['latency_p95_s']}s"
)
!python -m asksight_model.evaluate --checkpoint checkpoints/{VERSION} --questions-file evaluations/asksight_v1.jsonl
qs = json.load(open(f"checkpoints/{VERSION}/metrics.json"))["eval_asksight_asksight_v1"]
print(f"AskSight: sentence rate {qs['sentence_rate']:.1%}, mean words {qs['mean_words']}")
if "fact_recall" in qs:
    print(f"description fact recall {qs['fact_recall']:.1%} (heuristic; review predictions)")


## 5a. Review 50 answers

Prints 25 random test answers plus all 26 fixed AskSight cases. For each answer, note: complete sentence, correct, invented details, and the error type (color, text, count, description).


In [ ]:
# Print 25 random test answers plus the 26 fixed AskSight cases.
import json
import random

test = [json.loads(line) for line in open(f"checkpoints/{VERSION}/predictions_test.jsonl")]
fixed = [
    json.loads(line)
    for line in open(f"checkpoints/{VERSION}/predictions_asksight_asksight_v1.jsonl")
]

random.seed(0)
for row in random.sample(test, 25) + fixed:
    print("Q:", row["question"])
    print("A:", row["prediction"])
    print("Ref:", row.get("short_answer") or row.get("references"))
    print("Facts:", row.get("expected_facts"))
    print("-" * 60)


## 5b. Compare B and C

Run this after both checkpoints exist. Exact `vqa_accuracy` drops for sentence answers; compare `vqa_accuracy_contains` against the v0.3 baseline (**51.05%**).

| Check | Bar |
|---|---|
| `sentence_rate` (test) | ≥95% |
| `vqa_accuracy_contains` (test) | ≥51.05% |
| 50 reviewed answers | better than v0.3 |
| Invented details | fewer than v0.3 |
| Latency | under 3 s |

If B and C are close, choose **B**: it is simpler and faster. A small gap on 26 cases is noise.


In [ ]:
# Compare Run B and Run C against the v0.3 baseline.
import json
from pathlib import Path

runs = {
    "B (sentence, connector only)": "v0.4-b-sentence",
    "C (sentence + LoRA 16)": "v0.4-c-lora16",
}

print("v0.3 baseline: exact 49.97% | containment 51.05% | sentence rate 0%")
prompts = set()
for label, version in runs.items():
    path = Path(f"checkpoints/{version}/metrics.json")
    if not path.is_file():
        print(f"\n{label}: missing {path}; run that experiment first")
        continue
    metrics = json.loads(path.read_text())
    test = metrics["eval_test"]
    fixed = metrics.get("eval_asksight_asksight_v1", {})
    prompts.add(test["system_prompt_sha256"])
    print(f"\n{label}  ({version})")
    print(f"  test n: {test['n']}")
    print(f"  sentence rate: {test['sentence_rate']:.1%}")
    print(f"  exact VQA: {test['vqa_accuracy']:.1%}")
    print(f"  containment: {test['vqa_accuracy_contains']:.1%}")
    print(f"  mean words: {test['mean_words']}")
    print(f"  fixed sentence rate: {fixed.get('sentence_rate', 0):.1%}")
    print(f"  fixed direct containment: {fixed.get('direct_vqa_accuracy_contains', 0):.1%}")
    print(f"  latency median/p95: {test['latency_median_s']}s / {test['latency_p95_s']}s")

if len(prompts) > 1:
    print("\nWARNING: B and C used different system prompts; comparison is not clean.")

print("\nReview the 50 answers from 5a. If the runs are close, choose B.")


## 5c. Pretrained vision-language baseline

Evaluate Qwen3-VL-2B-Instruct on the same fixed 26 cases. It is an evaluation-only baseline, not a drop-in checkpoint for the current backend. This directly checks whether a model pretrained to connect images and language improves the app tasks. It downloads several GB of model files.


In [ ]:
%%time
!python -m asksight_model.evaluate_vlm --questions-file evaluations/asksight_v1.jsonl --output evaluations/runs/{VERSION}-qwen3-vl-2b-asksight-v1.jsonl


## 5d. Robustness (~3 min)

Re-runs the same held-out split under one realistic capture fault per condition — `blur`, `dark`, `obstructed` (a finger over the lens), `lowres` — and reports the accuracy drop against `clean`. This is the measured basis for the "please verify this" guidance the app shows for poor photos. The result lands in `metrics.json` under `eval_robustness`.


In [ ]:
%%time
!python -m asksight_model.robustness --checkpoint checkpoints/{VERSION} --split test --data-dir data/vqa


## 6. Download the checkpoint

Packages `checkpoints/<VERSION>/`. Colab storage is ephemeral — download the archive before the session ends. If you run 5c, zip the Qwen comparison files separately.


In [ ]:
!zip -qr {VERSION}-experiment.zip checkpoints/{VERSION}
import os
import shutil
zip_path = f"{VERSION}-experiment.zip"
if os.path.isdir("/kaggle/working"):  # Kaggle may also expose Colab compatibility variables
    output_path = f"/kaggle/working/{VERSION}-experiment.zip"
    if os.path.abspath(zip_path) != os.path.abspath(output_path):
        shutil.copy2(zip_path, output_path)
    zip_path = output_path
    size = os.path.getsize(zip_path) / 1e6
    print(f"ready for Kaggle's Output panel: {os.path.abspath(zip_path)} ({size:.1f} MB)")
elif os.environ.get("COLAB_RELEASE_TAG"):
    from google.colab import files
    files.download(zip_path)
else:
    size = os.path.getsize(zip_path) / 1e6
    print(f"ready to download: {os.path.abspath(zip_path)} ({size:.1f} MB)")


## 7. Serve the checkpoint (laptop / server)

```bash
cd ~/AskSight-checkpoints && unzip ~/Downloads/v0.2-20k-ckpt.zip
cd backend && source venv/bin/activate
pip install -e "../model[lora]"   # include LoRA when the checkpoint uses it
export ASKSIGHT_MODEL_CHECKPOINT=$HOME/AskSight-checkpoints/checkpoints/v0.2-20k
uvicorn app.main:app --reload   # /ask now returns real answers
```

For a larger experiment, set `TRAIN = 50000` and use a fresh `VERSION`. Compare on the same held-out data. Do not judge quality from answer length alone.
